# Tokenization
How text becomes tokens — BPE algorithm, vocabulary construction, and why tokenization matters for LLM behavior.

## 1. Theory & Intuition

### What is a Token?
A token is NOT a word — it's a subword unit that appears frequently enough in training data to get its own ID.

**Three approaches:**
| Approach | Example | Vocab Size | Sequence Length | Problem |
|----------|---------|------------|-----------------|---------|
| Character level | "cat" = ['c','a','t'] | ~256 | Very long | Too many tokens per word |
| Word level | "cat" = ['cat'] | 170,000+ | Short | Can't handle rare/new words |
| Subword (BPE) | "cat" = ['cat'] | ~50,000 | Balanced | Best of both worlds |

**Why " the" ≠ "the":**
Space is part of the token. "the" at word start vs middle = different frequencies = different tokens. This is why GPT tokenizers use Ġ (or similar) to mark word boundaries.

### BPE — Byte Pair Encoding Algorithm
Used by GPT, Llama, Mistral, and most modern LLMs.

**Steps:**
1. Start with character-level vocabulary
2. Count all adjacent character pairs in training corpus
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary reaches target size (e.g. 50,000 tokens)

**Key insight:** BPE learns morphology automatically — common words become single tokens, rare words split into meaningful subword pieces.

### Why Tokenization Matters for Interviews
- Token count affects cost and context limits
- Numbers tokenize badly — "1234" may be 3-4 tokens
- Non-English text uses more tokens per word
- Code tokenizes differently than prose
- Prompt length in tokens != prompt length in characters

## 2. Implementation — BPE From Scratch

In [ ]:
from collections import defaultdict

def get_vocab(corpus):
    """Split corpus into character-level tokens with word boundary marker"""
    vocab = defaultdict(int)
    for word in corpus.split():
        chars = ' '.join(list('Ġ' + word))  # Ġ marks word boundary (like GPT)
        vocab[chars] += 1
    return vocab

def get_pairs(vocab):
    """Count all adjacent pairs in vocabulary"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge the most frequent pair across all words"""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

# Train BPE on small corpus
corpus = "low low low lower lower newest newest highest"
vocab = get_vocab(corpus)

print("Initial vocabulary (character level):")
for word, freq in vocab.items():
    print(f"  '{word}' : {freq}")

print("\n" + "="*50)
print("Running 10 BPE merge operations...")
print("="*50)

merges = []
for i in range(10):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)
    print(f"\nMerge {i+1}: '{best_pair[0]}' + '{best_pair[1]}' → '{''.join(best_pair)}'")
    print(f"  Frequency: {pairs[best_pair]}")

print("\n" + "="*50)
print("Final learned merges (this IS the tokenizer):")
print("="*50)
for i, merge in enumerate(merges):
    print(f"  {i+1}. {merge[0]} + {merge[1]} → {''.join(merge)}")

## 3. Hands-on Practice — HuggingFace Tokenizers

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', '-q'], capture_output=True)

from transformers import AutoTokenizer

# Compare tokenization across different models
tokenizer_gpt2 = AutoTokenizer.from_pretrained('gpt2')

test_texts = [
    "Hello world",
    "tokenization",
    "ChatGPT is amazing",
    "1234567890",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "The bank can guarantee deposits",
]

print("GPT-2 Tokenization Analysis")
print("="*60)
for text in test_texts:
    tokens = tokenizer_gpt2.encode(text)
    token_strings = tokenizer_gpt2.convert_ids_to_tokens(tokens)
    print(f"\nText: '{text}'")
    print(f"Tokens ({len(tokens)}): {token_strings}")
    print(f"Token IDs: {tokens}")

In [ ]:
# Token counting — important for cost and context management
texts = {
    "Short sentence": "The cat sat on the mat.",
    "Technical text": "Implement a binary search algorithm with O(log n) time complexity.",
    "Code snippet": "for i in range(len(nums)): if nums[i] == target: return i",
    "Numbers": "The price is $1,234.56 and the date is 2024-01-15",
}

print("Token Count Analysis")
print("="*60)
for name, text in texts.items():
    tokens = tokenizer_gpt2.encode(text)
    chars = len(text)
    ratio = chars / len(tokens)
    print(f"\n{name}:")
    print(f"  Characters: {chars}")
    print(f"  Tokens: {len(tokens)}")
    print(f"  Chars per token: {ratio:.1f}")

print("\nKey insight: ~4 chars per token for English prose")
print("Numbers, code, and non-English text are less efficient")

## 4. Production Application

In [ ]:
# Production tokenization patterns
print("Production Tokenization Patterns")
print("="*60)

print("\n1. Counting tokens before API call (cost control):")
def count_tokens(text, tokenizer):
    return len(tokenizer.encode(text))

prompt = "Explain the transformer architecture in detail with examples."
token_count = count_tokens(prompt, tokenizer_gpt2)
print(f"   Prompt: '{prompt}'")
print(f"   Token count: {token_count}")
print(f"   Estimated cost at $0.01/1K tokens: ${token_count * 0.00001:.6f}")

print("\n2. Truncating to fit context window:")
def truncate_to_limit(text, tokenizer, max_tokens=100):
    tokens = tokenizer.encode(text)
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        return tokenizer.decode(tokens), True
    return text, False

long_text = "The transformer architecture " * 20
truncated, was_truncated = truncate_to_limit(long_text, tokenizer_gpt2, max_tokens=50)
print(f"   Was truncated: {was_truncated}")
print(f"   Truncated text: '{truncated[:80]}...'")

print("\n3. Special tokens in chat models:")
print("   [BOS] = beginning of sequence token")
print("   [EOS] = end of sequence token") 
print("   <|im_start|> = instruction start (ChatML format)")
print("   These add to your token count!")

## 5. Key Takeaways

### Summary
| Concept | Key Point |
|---------|-----------|
| Token | Subword unit, NOT a word |
| BPE | Iteratively merges most frequent character pairs |
| Word boundary | Space is part of the token — " the" ≠ "the" |
| Vocab size | ~50,000 for most modern LLMs |
| Efficiency | ~4 chars/token for English, less for numbers/code |

### Key Points
1. BPE learns morphology from frequency — no linguistic rules needed
2. Common words = single token, rare words = multiple tokens
3. Numbers tokenize poorly — "1234" can be 3-4 separate tokens
4. Code is more token-efficient than natural language
5. Always count tokens before API calls — affects cost and context limits
6. Special tokens ([BOS], [EOS], chat format tokens) add to your count
7. Non-English text uses more tokens per word than English

### Interview Question You Should Answer Cold
"How does BPE tokenization work and why does GPT tokenize ' the' differently from 'the'?"

Answer: BPE iteratively merges the most frequent adjacent character pairs in training data until reaching target vocabulary size. ' the' and 'the' appear in different positions with different frequencies, so they get merged into separate tokens. The space is part of the token, not a separator.